# 03. Integrated $L^p$ losses under incomplete sampling

**Status: preliminary supporting study.** Earlier three-seed clustered pilot and seed-zero uniform-versus-clustered GRU/Linear-Neural-CDE comparisons are complete. Missingness results form a separate pipeline check. Remaining core seeds are deliberately deferred because notebooks 05 and 06 test the same sampling-measure mechanism more directly and with complete three-seed evidence. Bibliographic entries are in `../papers/references.bib`.

Standard definitions, project choices and empirical conclusions are distinguished where they arise. Section 7.1 records the rationale and revision criterion for project choices.


## What to run

Pilot read by cells below, three paired seeds under each loss:

```bash
for loss in mse integral_l2; do
  for seed in 0 1 2; do
    python scripts/run_reconstruction.py \
      --config configs/baseline_${loss}.yaml \
      --out results/runs/loss_comparison/${loss}_seed${seed} \
      --seed ${seed}
  done
done
```

Core paired study of §10, one seed at a time:

```bash
python scripts/run_integral_study.py \
  --config configs/integral_core_study.yaml \
  --out results/runs/integral_core --seed 0
```

Grid crosses seed, architecture, target mechanism and loss; `--list` prints it. Paired members share paths, observations, initialisation, batch order and budget, so a difference between fits is attributable to training loss. Stored core evidence covers seed 0; seeds 1 and 2 are optional extensions. Final report claims exclude this branch. Missingness cell reads `robustness.json`, written by the same reconstruction runner when a config enables missingness evaluation.


## 1. Preliminary learning claim

**Motivation.** Notebook 01 derives empirical-sampling versus elapsed-time measures from standard quadrature and empirical-measure convergence. This experiment tests whether that distinction changes a finite learned predictor.

Notebook 01 establishes that unweighted target averages measure residual against empirical sampling measure, while quadrature measures residual against elapsed time. This notebook asks whether that distinction changes a finite learned predictor:

1. nonuniform target coverage increases the difference between predictors fitted with unweighted squared error and elapsed-time weighted $J_2$, across conventional and continuous-time architectures.

Each paired run shares paths, observations, model, initial parameters, batch order, optimiser, and training budget. Training objective is the varying factor. Exponent-specific robustness experiments are deferred until a downstream task requires them.


## 2. Generated paths and sampled data

**Source and choices.** Ornstein--Uhlenbeck process and Euler--Maruyama discretization are standard; see Kloeden and Platen (1992). Values of $\theta,\sigma,M$, split sizes and disjoint context/target construction are fixed for this study.

Let the fine reference grid be

$$
t_j=\frac{j}{M},\qquad j=0,\ldots,M,\qquad M=512.
$$

For each training example $i=1,\ldots,n_{\mathrm{train}}$, independently generate a univariate Ornstein–Uhlenbeck path $x^{(i)}:[0,1]\to\mathbb R$. Path index $(i)$ is a parenthesised superscript. Fine grid index $j$ and observation index $r$ are subscripts. Write $x^{(i)}_j:=x^{(i)}(t_j)$. Euler–Maruyama gives

$$
x^{(i)}_{j+1}
=x^{(i)}_j-\theta x^{(i)}_j\Delta t
+\sigma\sqrt{\Delta t}\,\varepsilon^{(i)}_j,\qquad
\varepsilon^{(i)}_j\overset{\mathrm{iid}}{\sim}\mathcal N(0,1),
$$

with $\theta=2$, $\sigma=0.5$, $x^{(i)}_0=1$, and $\Delta t=1/M$. Stored vector $(x^{(i)}_0,\ldots,x^{(i)}_M)$ is fine grid ground truth. Training uses its sampled context and target values.

For path $i$, choose two sorted, disjoint index sets:

$$
C^{(i)}=\{c^{(i)}_1<\cdots<c^{(i)}_C\},\qquad
T^{(i)}=\{q^{(i)}_1<\cdots<q^{(i)}_Q\},\qquad C^{(i)}\cap T^{(i)}=\varnothing,
$$

where $C=Q=64$ and $\{0,M\}\subset T^{(i)}$. Define context and target times for path $i$ by

$$
\tau^{(i)}_r:=t_{c^{(i)}_r},\qquad r=1,\ldots,C,
\qquad\text{and}\qquad
s^{(i)}_r:=t_{q^{(i)}_r},\qquad r=1,\ldots,Q.
$$

The context supplied to the model is

$$
\mathcal O^{(i)}=\Big((\tau^{(i)}_r,x^{(i)}(\tau^{(i)}_r),m^{(i)}_r)\Big)_{r=1}^{C},
$$

and target value is $y^{(i)}_r:=x^{(i)}(s^{(i)}_r)$ for $r=1,\ldots,Q$. Training targets are pairs $(s^{(i)}_r,y^{(i)}_r)$. Clean experiment has $m^{(i)}_r=1$. Disjoint index sets prevent direct copying.


## 3. Target sampling mechanisms

**Source and choices.** Empirical measure is standard. Uniform and exponential clustered index generators are study controls; time weighting for clumped observations has precedent in Rimoldini (2014). Informative-observation distinction follows Weaver, Xiao and Lu (2023).

For target times $s^{(i)}_1<\cdots<s^{(i)}_Q$, define empirical sampling measure and elapsed time measure

$$
\mu_Q^{(i)}=\frac1Q\sum_{r=1}^Q\delta_{s^{(i)}_r},
\qquad
\lambda(dt)=\frac{dt}{H^{(i)}},
\qquad H^{(i)}=s^{(i)}_Q-s^{(i)}_1.
$$

Uniform random selection has uniform expected time coverage. Clustered selection uses the same target count with nonuniform expected coverage. Context sampling remains uniform and fixed across target mechanisms, so model input does not change with target mechanism.

| target mechanism | controlled feature |
|---|---|
| uniform random selection | irregular gaps with uniform expected coverage |
| clustered selection | nonuniform time coverage at fixed sample count |

Informative sampling, where observation times depend on path values, is deferred because it requires an observation process model (Weaver, Xiao and Lu, 2023).

Pilot used the clustered law

$$
p_\beta(j)
=\frac{\exp\!\left(\beta(t_j-\tfrac12)\right)}
       {\sum_{k\in\mathcal E}\exp\!\left(\beta(t_k-\tfrac12)\right)}
$$

with $\beta=3$ for context and target selection. Parameter $\beta$ labels this generator and does not characterize general irregularity. Continuous $\beta$ sweep is removed from core design. Time-weighted estimation under clumped observations has direct precedent in Rimoldini (2014).


## 4. GRU input and output

**Architecture sources and choices.** Gated recurrent unit follows Cho et al. (2014). Fourier features follow Tancik et al. (2020). Time-gap and mask channels are standard irregular-series inputs, with GRU-D as related precedent (Che et al., 2018). Two-layer encoder, retained final state, decoder widths and frequencies are fixed study choices.

For a batch of $B$ paths, the model receives arrays with shapes

$$
t_{\mathrm{ctx}}\in\mathbb R^{B\times C},\quad
x_{\mathrm{ctx}},m_{\mathrm{ctx}}\in\mathbb R^{B\times C\times1},\quad
t_{\mathrm{query}}\in\mathbb R^{B\times Q}.
$$

For context step $r$, define the time gap $\Delta\tau^{(i)}_1=0$ and $\Delta\tau^{(i)}_r=\tau^{(i)}_r-\tau^{(i)}_{r-1}$ for $r>1$. The encoder input is

$$
v^{(i)}_r=\big(\tau^{(i)}_r,\Delta\tau^{(i)}_r,x^{(i)}(\tau^{(i)}_r),m^{(i)}_r\big)\in\mathbb R^4.
$$

A two layer GRU applies its recurrent update $h^{(i)}_r=\operatorname{GRU}_\theta(v^{(i)}_r,h^{(i)}_{r-1})$ and retains final hidden vector $h^{(i)}_C\in\mathbb R^{64}$. Thus one vector summarises all 64 context observations for path $i$.

A query time $s$ is represented by eight Fourier frequency pairs:

$$
\gamma(s)=\big(s,\sin(2^0\pi s),\cos(2^0\pi s),\ldots,\sin(2^7\pi s),\cos(2^7\pi s)\big).
$$

Decoder is a tanh network with two hidden layers of width 128. Its scalar prediction is

$$
\widehat x^{(i)}_\theta(s)=D_\theta\big(h^{(i)}_C,\gamma(s)\big).
$$

Decoder combines the path summary with each query time representation and returns an array in $\mathbb R^{B\times Q\times1}$. Training updates recurrent encoder and decoder jointly.


## 5. Integrated $L^p$ training objectives

This section combines established losses and quadrature with project use under irregular target sampling. Each objective is labelled separately.

For error $e^{(i)}_r=\widehat x^{(i)}_\theta(s^{(i)}_r)-y^{(i)}_r$, each objective specifies both a measure on target times and a penalty on residual magnitude. These choices have separate motivations.

### 5.1 Pointwise MSE

**Source and role.** Squared-error elicitation is standard; see Gneiting (2011). Equal weight over recorded targets is baseline in project proposal. Sampling-measure interpretation is established in notebook 01.

$$
L_{\mathrm{MSE}}(\theta)
=\frac1B\sum_{i=1}^B\frac1Q\sum_{r=1}^Q\left|e^{(i)}_r\right|^2.
$$

MSE treats each recorded target as one equally important supervised example. It is the standard baseline, has a smooth linear residual gradient, and elicits the conditional mean when a model estimates a random scalar target (Gneiting, 2011). Its time measure is empirical sampling measure $Q^{-1}\sum_r\delta_{s^{(i)}_r}$. Equal observation weight is appropriate when observations are units of interest. Under nonuniform time coverage it emphasizes regions containing more observations, irrespective of elapsed time represented by each observation. This sampling-measure effect is the mechanism tested here.

### 5.2 Elapsed-time $J_p$ family

**Source and choices.** Composite trapezoid rule is standard (Davis and Rabinowitz, 1984); time weighting under irregular sampling appears in Rimoldini (2014). Batch aggregation, omitted root and selected exponents are fixed below.

For sorted target times $s^{(i)}_1<\cdots<s^{(i)}_Q$, define normalised trapezoid weights

$$
w^{(i)}_1=\frac{s^{(i)}_2-s^{(i)}_1}{2H^{(i)}},\qquad
w^{(i)}_r=\frac{s^{(i)}_{r+1}-s^{(i)}_{r-1}}{2H^{(i)}}\ (1<r<Q),\qquad
w^{(i)}_Q=\frac{s^{(i)}_Q-s^{(i)}_{Q-1}}{2H^{(i)}},
$$

where $H^{(i)}=s^{(i)}_Q-s^{(i)}_1=1$ and $\sum_r w^{(i)}_r=1$. Weight concentration is summarised by

$$
n_{\mathrm{eff}}^{(i)}
=\frac{1}{\sum_{r=1}^Q\left(w^{(i)}_r\right)^2}.
$$

Equal weights give the maximum $n_{\mathrm{eff}}=Q$; weight concentration reduces it. A regular trapezoid grid lies slightly below $Q$ because its two endpoint weights are halved. For finite $p\geq1$, define

$$
J_p(\theta)
=\frac1B\sum_{i=1}^B\sum_{r=1}^Qw^{(i)}_r\left|e^{(i)}_r\right|^p
\approx\frac1B\sum_{i=1}^B\left\|e^{(i)}_\theta\right\|_{L^p}^p.
$$

Trapezoid weights are quadrature weights for integration with respect to elapsed time (Davis and Rabinowitz, 1984). Each interior residual represents half of the gap on either side. Time weighting of clumped observations also appears in Rimoldini (2014). Implementation omits the $p$th root. For one path and fixed $p$, root and power have the same minimiser; averaging paths before or after taking roots can change a fitted model.

### 5.3 $J_1$: integrated absolute error

Absolute error elicits conditional median and has linear residual influence (Gneiting, 2011). A contamination or asymmetric-noise study would be needed to make that distinction informative here.

$J_1$ weights error linearly and elicits a conditional median pointwise (Gneiting, 2011). Its influence does not grow with residual size, making it a candidate for contaminated or heavy-tailed targets. Clean Ornstein--Uhlenbeck conditional laws are symmetric, so conditional mean and median coincide; this dataset gives little reason to expect a systematic $J_1$ advantage. $J_1$ remains an evaluation metric until a contamination or asymmetric-noise condition supplies that motivation.

### 5.4 $J_2$: elapsed-time squared error

Hilbert-space role follows functional data analysis (Ramsay and Silverman, 2005). Holding quadratic exponent fixed while changing time measure isolates the sampling-measure mechanism.

$$
J_2(\theta)=\frac1B\sum_{i=1}^B\sum_{r=1}^Qw^{(i)}_r\left|e^{(i)}_r\right|^2.
$$

$J_2$ retains MSE's quadratic residual penalty while replacing empirical sampling measure by normalized elapsed time. It is therefore the controlled comparison for the sampling-measure claim: exponent, model and observations remain fixed. It approximates mean residual energy $\int|e(t)|^2dt$ and inherits Hilbert-space structure used in projection and functional regression (Ramsay and Silverman, 2005). On an evenly spaced grid it differs from MSE only through trapezoidal endpoint weights.

### 5.5 $J_4$: integrated fourth-power error

Contribution $a^4$ and derivative $4a^3$ follow directly from the definition. A peak-sensitive application would be needed to motivate this objective as a training loss.

$J_4$ gives residual magnitude $a$ a contribution proportional to $a^4$ and gradient magnitude proportional to $a^3$. It therefore concentrates optimization on large peaks. This is useful only when peak recovery has task value and is sensitive to outliers or corrupted targets. $J_4$ remains an evaluation metric until a local-peak task supplies that requirement.

### 5.6 $J_\infty$: worst sampled-time error

The supremum functional is used only for dense-grid evaluation, where it reports worst observed error.

Define

$$
J_\infty(\theta)
=\frac1B\sum_{i=1}^B\max_{1\leq r\leq Q}\left|e^{(i)}_r\right|.
$$

$J_\infty$ asks for uniform rather than average accuracy and detects a concentrated failure that contributes little to an integral. A sampled maximum can still miss errors inside long gaps, and its gradient is supplied only by a current maximizer. It is therefore a dense-grid diagnostic rather than a primary training objective in this study.


## 6. Sampling measure and exponent

Empirical-measure limit and quadrature distinction are developed in notebook 01. Conditional mean and median elicitation follows Gneiting (2011), $L^2$ functional geometry follows Ramsay and Silverman (2005), and rearrangement invariance follows Lieb and Loss (2001). Clean OU data supplies a controlled application of those facts.

Define residual path $e^{(i)}_\theta(t):=\widehat x^{(i)}_\theta(t)-x^{(i)}(t)$. Two independent choices define an integrated loss: measure on time and exponent on residual magnitude.

If empirical sampling measures converge to density $\rho$, unweighted $p$ power estimates

$$
\int_0^1 \left|e^{(i)}_\theta(t)\right|^p\rho(t)\,dt,
$$

while quadrature approximates

$$
\int_0^1 \left|e^{(i)}_\theta(t)\right|^p\,dt.
$$

Measure choice changes time emphasis. Exponent choice changes statistical target and tail emphasis. For $p>1$, unrestricted pointwise optimum $a_p(t)$ satisfies

$$
\mathbb E\!\left[
|X(t)-a_p(t)|^{p-2}(a_p(t)-X(t))
\mid\mathcal O
\right]=0.
$$

| exponent | pointwise target and emphasis |
|---:|---|
| $1$ | conditional median; linear outlier penalty |
| $2$ | conditional mean; quadratic penalty and Hilbert geometry |
| $4$ | centre weighted toward tail errors; strong peak penalty |
| $\infty$ | worst sampled time |

Absolute and squared losses elicit median and mean respectively (Gneiting, 2011). $L^2$ dominates functional data analysis because inner product structure supports bases, projection, and functional regression (Ramsay and Silverman, 2005).

Conditional Ornstein–Uhlenbeck distribution is Gaussian and symmetric. Its mean and median coincide, so clean OU data are a control where $p=1$ and $p=2$ have same unrestricted pointwise optimum. Exponent study needs contamination or asymmetric errors to expose robustness, and local pulses to expose peak emphasis.

Every raw $L^p$ objective compares aligned residual magnitudes. Measure preserving rearrangement of time leaves its value unchanged. Temporal alignment losses such as soft DTW address correspondence (Cuturi and Blondel, 2017); signatures address ordered path interactions.


## 7. Paired training and evaluation algorithm

**Comparison controls.** Shared data, initialization, batch order and fixed-budget evaluation isolate training objective. Exact seeds, splits, epochs and metrics are fixed below.

Planned studies use the following protocol for each seed $a$, data condition $d$, and objective $J_p$ under comparison.

1. Set the NumPy and PyTorch random seeds to $a$.
2. Generate fixed train, validation, and test splits containing 512, 128, and 256 independent base paths. Store fine grid truth for every path.
3. Generate one fixed context set for each path. Generate target sets according to condition $d$. Within $(a,d)$, paths, context, targets, and minibatch order are shared across objectives.
4. Initialise the same model parameters $\theta_0$ for every paired loss comparison. Create Adam with learning rate $10^{-3}$.
5. For epochs $k=1,\ldots,200$, draw a shared random permutation, divide it into batches of $B=64$ paths, predict at all target times, compute the chosen objective, back propagate, and take one Adam step.
6. Evaluate the final model once on 256 held out test paths.
7. On all 513 fine grid times, report $J_1$, $J_2$, $J_4$, and mean pathwise $J_\infty$ for every fitted model.
8. Compare objectives within seed, architecture, and target mechanism, then summarise paired differences across seeds.

The completed pilot predates this protocol. It uses 512 training and 128 validation paths, three seeds, one clustered mechanism, and no independent test split. Its values remain exploratory validation estimates.


### 7.1 Choice rationale and revision criteria

| project choice | rationale | check or revision criterion |
|---|---|---|
| Ornstein--Uhlenbeck generator with $\theta=2$, $\sigma=0.5$, $x_0=1$ | It provides reproducible continuous stochastic paths, known mean-reverting dynamics and Gaussian conditional laws. Symmetry makes it a control where mean and median coincide, separating sampling-measure question from robustness to asymmetric targets. Fixed initial value removes initial-condition heterogeneity. | Treat conclusions as mechanism evidence for this controlled process. Add another process only after core seeds finish or if model cannot learn held-out OU paths. |
| Euler--Maruyama fine grid with 513 points | Grid is eight times denser than 64 target observations and supplies common stored ground truth between samples. Power-of-two interval count aligns existing solvers and evaluations. Stored discretized path defines synthetic target, so current claim does not require convergence to exact OU sample path. | Describe generator as discretized OU. A coupled finer-grid study is required only before making a continuous-SDE claim. |
| 512/128/256 train/validation/test split | Training size gives eight batches of 64 per epoch; separate validation supports pipeline checks; test split remains untouched until design is fixed. Sizes keep CPU study feasible. | Increase test or seed count if paired uncertainty is too large for directional claim. Test split must not influence configuration choices. |
| 64 context and 64 disjoint targets including target endpoints | Equal counts hold information and supervision budgets fixed. Disjoint sets prevent direct copying. Target endpoints make $H^{(i)}=1$ and ensure every quadrature loss covers same horizon. | Verify disjointness, endpoints and counts in tests. Add sample-count sensitivity only after core comparison. |
| uniformly sampled context in core study | Context determines model information. Holding it fixed isolates target-loss measure from input missingness or coverage. | Run context robustness separately; any combined condition follows completion of isolated effects. |
| uniform versus clustered target mechanism at fixed count | Uniform random selection controls irregular gaps with uniform expected coverage. Clustered law with density bias $3$ creates unequal expected coverage without using path values, avoiding informative-sampling confound. | Record gap distribution and effective quadrature sample size. Replace bias if uniform and clustered coverage summaries are not materially separated. |
| GRU and parameter-matched Linear Neural CDE | GRU is conventional discrete sequential baseline; Neural CDE is continuous-time counterpart requested by project direction. Matching 65,537 versus 65,121 parameters reduces model-size explanation for architecture differences. | Require learning, gradient and held-out acceptance for each. Parameter count does not match optimization difficulty, so report architecture results separately. |
| final-state query decoder with eight Fourier frequencies | Final recurrent/CDE state summarizes context; query time permits predictions at arbitrary target points. Same decoder structure across architectures isolates encoder family. Fourier map is retained by repository adequacy test against raw scalar time. | Keep decoder identical across paired losses. Revisit architecture if both encoders share same time-localized failure. |
| MSE and $J_2$ primary objectives | Both penalize squared residuals. Their only intended difference is empirical target measure versus elapsed-time quadrature, matching core claim. | Uniform mechanism is near-equivalence control; larger clustered effect across seeds and architectures supports mechanism. |
| $J_1$, $J_4$ and $J_\infty$ as evaluation metrics | They reveal median/robustness, peak sensitivity and worst-time error without multiplying training comparisons before an application motivates those properties. | Promote one to training only with asymmetric contamination, peak-sensitive task or uniform-error requirement fixed in advance. |
| omitted $p$th root in $J_p$ | It avoids root singularity near zero and preserves minimizer for each individual path. Batch aggregation convention remains explicit because root before averaging would define another objective. | Keep convention fixed across runs; introduce rooted version only as separately named loss. |
| Adam, learning rate $10^{-3}$, batch 64, 200 epochs | Common optimizer and budget isolate objective. Batch 64 gives eight updates per epoch; existing pilot demonstrates learning at this budget. | Compare learning curves. Extend same paired budget for all losses if validation scores are still improving materially at epoch 200. |
| three paired seeds | Paired seeds measure initialization and batch-order variability within available CPU budget. Shared randomness reduces variance of loss contrasts. | Seeds 1 and 2 are mandatory. Increase count if paired effect changes sign or remains dominated by seed variation. |
| dense 513-point test evaluation | It scores recovery of stored underlying path on common time measure rather than only sampled training targets. This separates training emphasis from evaluation coverage. | Every model is scored under all metrics on identical grid. Add finer evaluation only if reference-grid check fails. |
| nested missingness masks at $0,0.1,0.3,0.5$ | Reusing uniforms makes degradation monotone in removed information at sample level and supports paired rate comparisons. Rates span mild to half-observed contexts without combining target changes. | Complete multiple seeds before robustness claims. Train-with-missingness is separate from clean-model stress test. |


## 8. Completed pilot configuration

Table records the implemented pilot configuration.

Pilot predates controlled design in §7.1. Its hyperparameters are exploratory engineering defaults and are not used to justify core claims; revised core study replaces clustered context, adds held-out test data and fixes architecture comparison.

| component | value |
|---|---|
| path process | univariate Ornstein–Uhlenbeck, $\theta=2$, $\sigma=0.5$, and $x^{(i)}(0)=1$ for every path $i$ |
| fine grid | 513 points on $[0,1]$ |
| data | 512 training paths, 128 validation paths |
| samples per path | 64 context, 64 disjoint targets including endpoints |
| sampling | clustered, $\beta=3$; both context and target affected |
| corruption | none |
| model | two layer GRU, hidden width 64; tanh decoder width 128; 8 Fourier frequencies |
| optimisation | Adam, learning rate $10^{-3}$, batch 64, 200 epochs |
| paired seeds | 0, 1, 2 |


In [1]:
import json
from pathlib import Path
from statistics import mean
import pandas as pd
from IPython.display import display

root = Path('../results/runs/loss_comparison')
pilot = {}
for path in sorted(root.glob('*_seed*/final.json')):
    loss_name, seed = path.parent.name.rsplit('_seed', 1)
    pilot[(loss_name, int(seed))] = json.loads(path.read_text())

metrics = ['fine_mse', 'fine_integral_l2', 'fine_integral_l1', 'fine_integral_linf']
summary = {
    loss: {metric: mean(pilot[(loss, seed)][metric] for seed in range(3)) for metric in metrics}
    for loss in ['mse', 'integral_l2']
}
paired_primary = [
    {
        'seed': seed,
        'mse_trained': pilot[('mse', seed)]['fine_integral_l2'],
        'weighted_l2_trained': pilot[('integral_l2', seed)]['fine_integral_l2'],
        'relative_improvement_percent': 100 * (
            pilot[('mse', seed)]['fine_integral_l2']
            - pilot[('integral_l2', seed)]['fine_integral_l2']
        ) / pilot[('mse', seed)]['fine_integral_l2'],
    }
    for seed in range(3)
]
print('Mean fine-grid validation metrics; source:', root)
display(pd.DataFrame(summary).T)
print('Paired fine-grid squared L2 comparison')
display(pd.DataFrame(paired_primary).set_index('seed'))


Mean fine-grid validation metrics; source: ../results/runs/loss_comparison


,fine_mse,fine_integral_l2,fine_integral_l1,fine_integral_linf
mse,0.008476,0.008480,0.071919,0.250602
integral_l2,0.008135,0.008133,0.071366,0.245308


Paired fine-grid squared L2 comparison


,mse_trained,weighted_l2_trained,relative_improvement_percent
seed,,,
0,0.008307,0.007996,3.740188
1,0.008372,0.008459,-1.043742
2,0.008763,0.007944,9.339697


## 9. Pilot results

Preceding cell calculates the exact mean metrics and seedwise contrasts from stored paired runs in `results/runs/loss_comparison/`; its displayed tables are the numerical record. They are exploratory because pilot lacks an independent test split. Weighted training lowers the across-seed mean primary score, although the seedwise direction is inconsistent. Stored runs yield different fitted solutions under the two objectives. The sampling mechanism responsible for the difference and any general performance ordering remain open.


## 10. Core architecture and sampling study

GRU and Neural CDE are established model families (Cho et al., 2014; Kidger et al., 2020). This comparison fixes parameter counts, paired contrast $\Delta_{a,m}$ and sampling controls.

Hold uniformly sampled context and $Q=64$ targets fixed. Compare uniform random target selection with clustered target selection. For each mechanism, fit paired models with unweighted squared error and time weighted $J_2$. Run both GRU and parameter matched Linear Neural CDE architectures over three seeds. GRU has 65,537 parameters and Neural CDE has 65,121.

For seed $a$ and mechanism $m$, define

$$
\Delta_{a,m}
=R_{a,m}(J_2\text{ training})
-R_{a,m}(\text{unweighted squared training}),
$$

where $R$ is fine grid $J_2$ on 256 held out test paths. Negative values favour time weighting. Uniform target selection is the coverage control. Clustered target selection tests the sampling-measure mechanism. A larger weighted-loss effect under clustering, reproduced across architectures and seeds, supports that mechanism.

### 10.1 Seed 0 pipeline result

Following cell reads every value from `results/runs/integral_core/*_seed0/test.json`, with run identity and parameter count from corresponding `meta.json`, and displays the complete numerical table. Weighted training improves every seed-zero comparison, with a larger relative change under clustered targets for both architectures. One seed verifies the revised pipeline and supplies a directional result; it cannot support a stable architecture-level claim. Later three-seed evidence in notebooks 05 and 06 tests the sampling-measure mechanism used in the final report.

### 10.2 Deferred exponent extensions

$J_1$ and $J_4$ remain implemented and reported. Separate contamination and local-peak experiments are deferred because they answer application-specific robustness questions rather than the current sampling-measure claim. Reintroduce an exponent study only when data or a downstream task supplies that requirement.


In [2]:
core_root = Path('../results/runs/integral_core')
core_rows = []
for test_path in sorted(core_root.glob('*_seed*/test.json')):
    meta_path = test_path.with_name('meta.json')
    report = json.loads(test_path.read_text())
    meta = json.loads(meta_path.read_text())
    job = meta['job']
    core_rows.append({
        'architecture': job['model']['name'],
        'target mechanism': job['mechanism']['name'],
        'loss': job['loss'],
        'seed': job['seed'],
        'parameters': meta['parameters'],
        'test J2': report['fine_integral_l2'],
        'test MSE': report['fine_mse'],
        'test J1': report['fine_integral_l1'],
        'test J4': report['fine_integral_l4'],
        'test Linf': report['fine_integral_linf'],
        'source': str(test_path),
    })
core_results = pd.DataFrame(core_rows)
if core_results.empty:
    print('Core outputs pending:', core_root)
else:
    display(core_results.sort_values(['seed', 'architecture', 'target mechanism', 'loss']))
    paired_core = core_results.pivot(
        index=['seed', 'architecture', 'target mechanism'],
        columns='loss', values='test J2',
    )
    paired_core['relative decrease from J2 training'] = (
        paired_core['mse'] - paired_core['integral_l2']
    ) / paired_core['mse']
    display(paired_core)


,architecture,target mechanism,loss,seed,parameters,test J2,test MSE,test J1,test J4,test Linf,source
0,gru,clustered,integral_l2,0,65537,0.006594,0.006595,0.063995,0.000141,0.226379,../results/runs/integral_core/gru_clustered_in...
1,gru,clustered,mse,0,65537,0.007285,0.007280,0.066763,0.000180,0.234875,../results/runs/integral_core/gru_clustered_ms...
2,gru,uniform,integral_l2,0,65537,0.006373,0.006377,0.063107,0.000130,0.221259,../results/runs/integral_core/gru_uniform_inte...
3,gru,uniform,mse,0,65537,0.006645,0.006647,0.064517,0.000139,0.225947,../results/runs/integral_core/gru_uniform_mse_...
4,linear_cde,clustered,integral_l2,0,65121,0.007865,0.007858,0.070514,0.000189,0.238884,../results/runs/integral_core/linear_cde_clust...
5,linear_cde,clustered,mse,0,65121,0.009395,0.009386,0.076980,0.000273,0.257110,../results/runs/integral_core/linear_cde_clust...
6,linear_cde,uniform,integral_l2,0,65121,0.007459,0.007453,0.068673,0.000170,0.229575,../results/runs/integral_core/linear_cde_unifo...
7,linear_cde,uniform,mse,0,65121,0.007768,0.007760,0.070061,0.000184,0.236165,../results/runs/integral_core/linear_cde_unifo...


loss                                integral_l2       mse  \
seed architecture target mechanism                          
0    gru          clustered            0.006594  0.007285   
                  uniform              0.006373  0.006645   
     linear_cde   clustered            0.007865  0.009395   
                  uniform              0.007459  0.007768   

loss                                relative decrease from J2 training  
seed architecture target mechanism                                      
0    gru          clustered                                   0.094849  
                  uniform                                     0.041068  
     linear_cde   clustered                                   0.162893  
                  uniform                                     0.039731

## 11. Separate context missingness study

Value-mask inputs have precedent in missing-time-series models such as GRU-D (Che et al., 2018). Reused uniforms produce nested masks; zero filling, fixed clean model and chosen rates isolate loss of context information.

Target sampling changes the measure used by the training loss. Context missingness changes the information supplied to the model. It is therefore a separate input robustness study. For each clean context value draw one fixed variable $U^{(i)}_r\sim\operatorname{Uniform}(0,1)$ and, at rate $\rho$, set

$$
m^{(i)}_r(\rho)=\mathbf 1\{U^{(i)}_r\geq\rho\},\qquad
\widetilde x^{(i)}_r(\rho)=m^{(i)}_r(\rho)x^{(i)}(\tau^{(i)}_r).
$$

For path $i$, model receives zero filled values $\widetilde x^{(i)}_r(\rho)$ and masks $m^{(i)}_r(\rho)$. Reusing $U^{(i)}_r$ across rates makes masks nested: every value missing at 10% remains missing at 30% and 50%. Query targets and fine grid ground truth stay fixed.

Robustness experiment first evaluates fixed clean trained models at $\rho\in\{0,0.1,0.3,0.5\}$ across paired seeds. A separate experiment trains with missingness augmentation.


In [3]:
missingness_path = Path('../results/runs/missingness_mse_seed0/robustness.json')
missingness_check = json.loads(missingness_path.read_text())['missingness']
missingness_rows = [
    {
        'missing rate': float(rate),
        'fine-grid squared L2': values['fine_integral_l2'],
        'mean pathwise Linf': values['fine_integral_linf'],
    }
    for rate, values in missingness_check.items()
]
print('source:', missingness_path)
display(pd.DataFrame(missingness_rows).set_index('missing rate'))


source: ../results/runs/missingness_mse_seed0/robustness.json


,fine-grid squared L2,mean pathwise Linf
missing rate,,
0.0,0.008307,0.244691
0.1,0.009616,0.260945
0.3,0.017053,0.319491
0.5,0.027741,0.375408


### Exploratory pipeline check

Preceding cell reads `results/runs/missingness_mse_seed0/robustness.json` and displays every recorded rate and metric. Error increases as context values are removed. One seed verifies masking and evaluation plumbing; stable degradation estimates and loss or architecture comparisons require paired seeds.


## 12. Reproducibility map

Paths below identify code, configurations and stored outputs supporting notebook statements.

- Path generation and index sampling: `src/pathloss/paths.py` and `src/pathloss/datasets.py`
- GRU query model: `src/pathloss/models.py`
- MSE, trapezoid weights, and squared $L^2$: `src/pathloss/losses.py`
- Training and evaluation loop: `src/pathloss/train.py`
- Completed paired outputs: `results/runs/loss_comparison/`
- Exploratory missingness output: `results/runs/missingness_mse_seed0/robustness.json`

Core study runner: `scripts/run_integral_study.py`; configuration: `configs/integral_core_study.yaml`; ARC array: `scripts/arc/submit_integral_core_array.slurm`; seed 0 outputs: `results/runs/integral_core/`. Later three-seed evidence is in notebook 05 for a fixed target and notebook 06 for a stream-to-stream operator; this preliminary branch is not used to support cross-seed report claims.
